# ASTRA · Notebook 00 — The datasets: an audit in figures

**CS787 (Generative AI) project, Group 4.** This notebook needs **no GPU** (set Accelerator to *None*; it costs no GPU quota) and runs in about 10 minutes.

**Goal.** Show what the data look like and check, with numbers, the reasons behind the choices of the benchmark notebooks (03–06). Each section answers one question:

| Section | Question | Used for |
|---|---|---|
| 1 | Which datasets, how many curves, how many classes? | the data section of the report |
| 2 | How unbalanced are the classes, and which are rare? What does a curve of each class look like? What does pre-processing do to a raw star? | motivation: rare classes |
| 3 | What do period and amplitude say about the class? How irregular is the sampling in phase? How noisy are the curves? | the conditions given to the models; the amplitude-normalised target; irregular sampling |
| 4 | Do curves of one star sit on both sides of a split? How finely can each test set measure a class? | why the benchmark splits over stars and tests on 20% of them |
| 5 | Which classes does a classifier trained on real data confuse? | where augmentation has room to help |
| 6 | What is in the PELS-VAE training file (OGLE-III)? | the leakage found in notebook 02 |

**Data.** The pre-processed files of notebook 01: `catalina_L100.npz` (built here from the public Catalina release if it is not attached, about 3 minutes) and `ztf_L40.npz` (attach the dataset that holds it; without it the ZTF parts are skipped). Section 6 downloads the PELS-VAE file from Zenodo (296 MB).

**How to run.** Accelerator *None*, Internet *On*, Run All. Figures go to `figures_data/`, tables to `tables_data/`.

In [ ]:
# ---- Configuration (CPU only) ----
import os, sys, json, time, shutil, traceback
from pathlib import Path

DATASETS = ["catalina", "ztf"]      # ztf is skipped if ztf_L40.npz is not attached under Add Input
WITH_OGLE = True                    # section 6: the PELS-VAE training file (296 MB download)
WITH_PIPELINE = True                # section 2: the raw-to-input figure (needs the raw Catalina photometry, 271 MB)
SEED = 0                            # the split shown is the benchmark's split 0
L = {"catalina": 100, "ztf": 40}
CLASS_RANGE_K, DATA_SEED = 2.0, 0   # Catalina pre-processing, as in notebook 01

RUN_STARTED = time.time()
WORK = Path(os.environ.get("ASTRA_WORK", "/kaggle/working"))
INPUT = Path(os.environ.get("ASTRA_INPUT", "/kaggle/input"))
FIGS, TABLES = WORK / "figures_data", WORK / "tables_data"
for d in (FIGS, TABLES, WORK / "data", WORK / "astra"):
    d.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
sys.path.insert(0, str(WORK))


def safely(what, fn):
    """One failing figure must not stop the rest of the audit: report it and go on."""
    try:
        return fn()
    except Exception:
        print(f"\n!! {what} failed:\n{traceback.format_exc()}")


def show(title, table, name, index=False):
    if table is not None and len(table):
        table.to_csv(TABLES / name, index=index)
        print(f"\n{title}\n{table.to_string(index=index)}")

In [ ]:
%%writefile astra/__init__.py
"""ASTRA: generative models for variable-star light curves (CS787 project, Group 4)."""


In [ ]:
%%writefile astra/common.py
"""Small helpers shared by every notebook: seeds, paths, time guard, safe saving, logging."""
import json
import os
import random
import time
import urllib.request
from pathlib import Path

import numpy as np
import torch

WORK = Path(os.environ.get("ASTRA_WORK", "/kaggle/working"))
INPUT = Path(os.environ.get("ASTRA_INPUT", "/kaggle/input"))


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def save_json(obj, path):
    """Write to a temp file first, so a kill mid-write never leaves a broken file."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, indent=1, default=float), encoding="utf-8")
    os.replace(tmp, path)


def load_json(path, default=None):
    path = Path(path)
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else default


def torch_save(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)


def restore_rng(state, device):
    """Put back the random-number state saved in a resumable checkpoint. torch.load(map_location=GPU) moves
    the saved states to the GPU, but the generators only accept CPU tensors, so they are moved back first.
    Never fatal: without it a resumed run just continues with fresh random numbers. Returns True on success."""
    try:
        torch.set_rng_state(state["rng_cpu"].cpu())
        if state.get("rng_cuda") is not None and str(device).startswith("cuda"):
            torch.cuda.set_rng_state_all([s.cpu() for s in state["rng_cuda"]])
        return True
    except Exception as e:
        print(f"could not restore the random-number state ({type(e).__name__}: {e}); continuing with a fresh one",
              flush=True)
        return False


class Deadline:
    """Session time guard: Kaggle kills a session at 12 h, we stop cleanly before that."""

    def __init__(self, deadline_unix):
        self.t = float(deadline_unix)

    def left(self):
        return self.t - time.time()

    def passed(self, reserve_s=0.0):
        return self.left() < reserve_s


class Logger:
    def __init__(self, path):
        self.path = Path(path)
        self.path.parent.mkdir(parents=True, exist_ok=True)

    def __call__(self, *msg):
        line = time.strftime("%H:%M:%S ") + " ".join(str(m) for m in msg)
        print(line, flush=True)
        with open(self.path, "a", encoding="utf-8") as f:
            f.write(line + "\n")


def find_input(name):
    """A file attached under /kaggle/input (e.g. a previous version's output), if any."""
    hits = sorted(INPUT.glob(f"**/{name}")) if INPUT.exists() else []
    return hits[0] if hits else None


def download(url, dest, tries=5):
    """Download once; reuse a copy attached under /kaggle/input if there is one."""
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    attached = find_input(dest.name)
    if attached is not None:
        return attached
    dest.parent.mkdir(parents=True, exist_ok=True)
    for i in range(tries):
        try:
            tmp = dest.with_suffix(dest.suffix + ".part")
            urllib.request.urlretrieve(url, tmp)
            os.replace(tmp, dest)
            return dest
        except Exception as e:  # network hiccup: wait and retry
            print(f"download failed ({e}), retry {i + 1}/{tries}")
            time.sleep(10 * (i + 1))
    raise RuntimeError(f"could not download {url}")


def peak_gpu_mem_gb():
    if not torch.cuda.is_available():
        return 0.0
    return torch.cuda.max_memory_allocated() / 1e9


In [ ]:
%%writefile astra/catalina.py
"""Catalina Surveys DR1 periodic variables, pre-processed exactly as in Garcia-Jara et al. (2022), Sec. 3.

Steps: period folding -> outlier filtering (sliding z-score, then a class-range filter) -> keep curves with
at least L points -> random sub-sampling to L points -> median centering. Then the paper's split: validation
and test each hold 20% of the smallest class, for every class (Sec. 4.1).
"""
from pathlib import Path

import numpy as np
import pandas as pd

from .common import download

CATALOG_URL = "http://nesssi.cacr.caltech.edu/DataRelease/CatalinaVars.tbl"
PHOT_URL = "http://nesssi.cacr.caltech.edu/DataRelease/AllVar.phot.gz"

# Table 1 of the paper: Drake et al. (2014) type id -> new class
CLASS_MAP = {1: "EBSD/D", 3: "EBSD/D",                      # EW, beta Lyrae
             4: "RRL", 5: "RRL", 6: "RRL", 7: "RRL",        # RRab, RRc, RRd, Blazhko
             2: "EBC",                                      # EA
             13: "LPV",
             11: "DSCT", 12: "DSCT",                        # HADS, LADS
             9: "CEP", 10: "CEP"}                           # ACEP, Cep-II
CLASSES = ["EBSD/D", "RRL", "EBC", "LPV", "DSCT", "CEP"]
PAPER_COUNTS = {"EBSD/D": 28980, "RRL": 7533, "EBC": 4500, "LPV": 483, "DSCT": 241, "CEP": 182}   # Table 2
TRAIN, VAL, TEST = 0, 1, 2


def read_catalog(path):
    cols = ["css_id", "numid", "ra", "dec", "vmag", "period", "amp_cat", "nobs", "vartype"]
    cat = pd.read_csv(path, sep=r"\s+", comment="#", header=None, names=cols)
    cat = cat[cat.vartype.isin(list(CLASS_MAP))].copy()
    cat["period"] = pd.to_numeric(cat.period)          # "\nodata" only occurs in type 17, dropped above
    cat["cls"] = cat.vartype.map(CLASS_MAP)
    return cat.sort_values("numid").reset_index(drop=True)


def read_photometry(path, numids):
    """All (MJD, mag, err) rows of the wanted objects, grouped per object."""
    ph = pd.read_csv(path, header=None, usecols=[0, 1, 2, 3], names=["numid", "mjd", "mag", "err"],
                     dtype={"numid": np.int64, "mjd": np.float64, "mag": np.float64, "err": np.float64})
    ph = ph[ph.numid.isin(set(numids))].sort_values(["numid", "mjd"], kind="stable")
    ids, start = np.unique(ph.numid.values, return_index=True)
    end = np.append(start[1:], len(ph))
    t, m, e = ph.mjd.values, ph.mag.values, ph.err.values
    return {i: (t[a:b], m[a:b], e[a:b]) for i, a, b in zip(ids, start, end)}


def fold(t, period):
    """Eqs. 8-9: phase = (t mod T) / T, in [0, 1)."""
    return np.mod(t, period) / period


def zscore_keep(m, ws=20, z=3.0, passes=2):
    """Sliding-window z-score filter on a phase-sorted curve (window 20, drop z > 3, two passes)."""
    keep = np.ones(len(m), bool)
    for _ in range(passes):
        s = pd.Series(m[keep])
        mu = s.rolling(ws, center=True, min_periods=2).mean().values
        sd = s.rolling(ws, center=True, min_periods=2).std().values
        with np.errstate(invalid="ignore", divide="ignore"):
            bad = np.abs(s.values - mu) / sd > z
        idx = np.flatnonzero(keep)
        keep[idx[bad]] = False
    return keep


def preprocess(data_dir, L=100, class_range_k=3.0, seed=0, log=print):
    """Returns a dict of arrays (one row per kept star) plus a per-class count table."""
    data_dir = Path(data_dir)
    cat = read_catalog(download(CATALOG_URL, data_dir / "CatalinaVars.tbl"))
    log(f"catalog: {len(cat):,} stars in the 6 classes")
    phot = read_photometry(download(PHOT_URL, data_dir / "AllVar.phot.gz"), cat.numid.values)
    log(f"photometry found for {len(phot):,} of them")

    # 1-2. fold, sort by phase, sliding z-score filter
    curves = []
    for r in cat.itertuples():
        if r.numid not in phot:
            continue
        t, m, e = phot[r.numid]
        ph = fold(t, r.period)
        o = np.argsort(ph, kind="stable")
        ph, m, e = ph[o], m[o], e[o]
        k = zscore_keep(m)
        curves.append((r.numid, r.cls, r.period, ph[k], m[k], e[k]))

    # 2b. class-range filter: drop a curve if > 90% of its magnitudes lie outside median +- k*std of its class
    stats = {}
    for c in CLASSES:
        allm = np.concatenate([cv[4] for cv in curves if cv[1] == c])
        stats[c] = (np.median(allm), np.std(allm))
    kept, n_range, n_short = [], 0, 0
    for cv in curves:
        med, sd = stats[cv[1]]
        if np.mean(np.abs(cv[4] - med) > class_range_k * sd) > 0.9:
            n_range += 1
        elif len(cv[4]) < L:       # 3. need at least L points to sub-sample
            n_short += 1
        else:
            kept.append(cv)
    log(f"dropped {n_range} by the class-range filter and {n_short} with fewer than {L} points")

    # 4-5. random sub-sampling to L points (kept in phase order), then median centering
    rng = np.random.default_rng(seed)
    n = len(kept)
    out = dict(numid=np.zeros(n, np.int64), y=np.zeros(n, np.int64), period=np.zeros(n),
               phase=np.zeros((n, L), np.float32), mag=np.zeros((n, L), np.float32),
               err=np.zeros((n, L), np.float32), amp=np.zeros(n, np.float32))
    for i, (numid, c, period, ph, m, e) in enumerate(kept):
        s = np.sort(rng.choice(len(m), L, replace=False))
        x = m[s] - np.median(m[s])
        out["numid"][i], out["y"][i], out["period"][i] = numid, CLASSES.index(c), period
        out["phase"][i], out["mag"][i], out["err"][i] = ph[s], x, e[s]
        out["amp"][i] = (x.max() - x.min()) / 2          # half peak-to-peak of the L points

    # split (Sec. 4.1): val and test each get 20% of the smallest class, for every class
    counts = np.bincount(out["y"], minlength=len(CLASSES))
    n_hold = int(0.2 * counts.min())
    split = np.full(n, TRAIN, np.int8)
    for c in range(len(CLASSES)):
        idx = rng.permutation(np.flatnonzero(out["y"] == c))
        split[idx[:n_hold]] = VAL
        split[idx[n_hold:2 * n_hold]] = TEST
    out["split"] = split
    out["classes"] = np.array(CLASSES)
    table = pd.DataFrame({"class": CLASSES, "ours": counts,
                          "paper": [PAPER_COUNTS[c] for c in CLASSES]})
    table["ours/paper"] = (table.ours / table.paper).round(3)
    table["train"] = [int(((out["y"] == c) & (split == TRAIN)).sum()) for c in range(len(CLASSES))]
    table["val"] = n_hold
    table["test"] = n_hold
    return out, table


In [ ]:
%%writefile astra/sampling.py
"""The resampling block of Garcia-Jara et al. (2022), Sec. 4.6.

Each class keeps its own shuffled queue. A class is picked with probability
    p_i = gamma / N + (1 - gamma) * S_i / S          (Eq. 12)
and the next sample is taken from its queue; an empty queue is reshuffled.
gamma = 0 keeps the original class mix, gamma = 1 gives a uniformly balanced stream.
"""
import numpy as np


class ClassResampler:
    def __init__(self, y, gamma, seed=0, n_classes=None):
        y = np.asarray(y)
        self.n_classes = n_classes or int(y.max()) + 1
        self.rng = np.random.default_rng(seed)
        self.pools = [np.flatnonzero(y == c) for c in range(self.n_classes)]
        sizes = np.array([len(p) for p in self.pools], float)
        self.p = gamma / self.n_classes + (1 - gamma) * sizes / sizes.sum()
        self.p[sizes == 0] = 0
        self.p /= self.p.sum()
        self.queues = [self.rng.permutation(p) for p in self.pools]
        self.pos = np.zeros(self.n_classes, int)

    def _take(self, c, k):
        out = []
        while k > 0:
            if self.pos[c] == len(self.queues[c]):
                self.queues[c] = self.rng.permutation(self.pools[c])
                self.pos[c] = 0
            n = min(k, len(self.queues[c]) - self.pos[c])
            out.append(self.queues[c][self.pos[c]:self.pos[c] + n])
            self.pos[c] += n
            k -= n
        return np.concatenate(out)

    def sample(self, batch):
        """Indices of one batch (classes drawn with p, samples without replacement inside a class)."""
        cls = self.rng.choice(self.n_classes, size=batch, p=self.p)
        idx = np.empty(batch, np.int64)
        for c in np.unique(cls):
            where = np.flatnonzero(cls == c)
            idx[where] = self._take(c, len(where))
        return idx


In [ ]:
%%writefile astra/classifier.py
"""The downstream classifier of Garcia-Jara et al. (2022), Sec. 4.3 and Table 3.

Base classifier: 5 conv blocks (Conv k=3 + BatchNorm, MaxPool 2 + ReLU), channels 2->32->64->128->64->64,
then Dense -> 100, BN + ReLU + Dropout, Dense -> N. Input = magnitudes and phases (2 channels).
The classifier is an ensemble of 5 base CNNs, and every score is repeated over 15 independent runs,
i.e. 75 small CNNs per table row. We train them side by side as ONE network with a leading "member"
axis: every member has its own weights and its own mini-batches, so they are exactly as independent
as 75 separate trainings. Each layer is one batched matrix product over the members (a convolution
with kernel 3 is a matrix product on the 3 shifted copies of its input), which a GPU runs far faster
than 75 small networks or a grouped convolution (measured on a T4: 133 s per epoch with grouped
convolutions). Mixed precision (fp16) is used on the GPU for the same reason.
"""
import math

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from .sampling import ClassResampler


class MemberLinear(nn.Module):
    """M independent Linear layers. x: (M, B, fin) -> (M, B, fout)."""

    def __init__(self, M, fin, fout):
        super().__init__()
        bound = 1 / math.sqrt(fin)                     # same init as nn.Linear
        self.w = nn.Parameter(torch.empty(M, fin, fout).uniform_(-bound, bound))
        self.b = nn.Parameter(torch.empty(M, 1, fout).uniform_(-bound, bound))

    def forward(self, x):
        return torch.baddbmm(self.b, x, self.w)


class MemberConv(nn.Module):
    """M independent Conv1d(kernel 3, padding 1) layers. x: (M, cin, B, L) -> (M, cout, B, L)."""

    def __init__(self, M, cin, cout, k=3):
        super().__init__()
        self.k = k
        bound = 1 / math.sqrt(cin * k)                 # same init as nn.Conv1d
        self.w = nn.Parameter(torch.empty(M, cout, cin * k).uniform_(-bound, bound))
        self.b = nn.Parameter(torch.empty(M, cout, 1).uniform_(-bound, bound))

    def forward(self, x):
        M, C, B, L = x.shape
        xp = F.pad(x, (self.k // 2, self.k // 2))
        cols = torch.stack([xp[..., i:i + L] for i in range(self.k)], 2)       # (M, cin, k, B, L)
        return torch.baddbmm(self.b, self.w, cols.reshape(M, C * self.k, B * L)).view(M, -1, B, L)


class CNNEnsemble(nn.Module):
    """M copies of the paper's base classifier (Table 3), run side by side. Input (M, B, in_ch, L)."""

    def __init__(self, M, n_classes, L, in_ch=2, dropout=0.5):
        super().__init__()
        self.M, self.in_ch = M, in_ch
        chans = [in_ch, 32, 64, 128, 64, 64]
        self.convs = nn.ModuleList(MemberConv(M, a, b) for a, b in zip(chans[:-1], chans[1:]))
        self.bns = nn.ModuleList(nn.BatchNorm2d(M * b) for b in chans[1:])
        l = L
        for _ in range(5):
            l = math.ceil(l / 2)                       # ceil(L/32) as in Table 3
        self.fc1 = MemberLinear(M, 64 * l, 100)
        self.bn = nn.BatchNorm1d(M * 100)
        self.drop = nn.Dropout(dropout)
        self.fc2 = MemberLinear(M, 100, n_classes)

    def features(self, x):
        """Output of the last conv block, flattened: (M, B, 64 * ceil(L/32))."""
        h = x.transpose(1, 2)                                              # (M, C, B, L)
        for conv, bn in zip(self.convs, self.bns):
            h = conv(h)
            M, C, B, L = h.shape
            h = bn(h.reshape(1, M * C, B, L))                              # statistics over (B, L) per member-channel
            h = F.relu(F.max_pool2d(h, (1, 2), ceil_mode=True)).view(M, C, B, -1)
        return h.transpose(1, 2).reshape(h.shape[0], h.shape[2], -1)

    def forward(self, x):
        h = self.fc1(self.features(x))                                     # (M, B, 100)
        M, B, _ = h.shape
        h = self.bn(h.transpose(1, 2).reshape(1, M * 100, B)).view(M, 100, B).transpose(1, 2)
        return self.fc2(self.drop(F.relu(h)))                              # (M, B, n_classes) logits


def reference_cnn(ens, m):
    """Member m of an ensemble rebuilt from standard PyTorch layers (used only by the self-tests)."""
    layers = []
    for conv in ens.convs:
        cout, cin = conv.w.shape[1], conv.w.shape[2] // conv.k
        ref = nn.Conv1d(cin, cout, conv.k, padding=conv.k // 2)
        ref.weight.data, ref.bias.data = conv.w[m].reshape(cout, cin, conv.k).clone(), conv.b[m, :, 0].clone()
        layers += [ref, nn.BatchNorm1d(cout), nn.MaxPool1d(2, ceil_mode=True), nn.ReLU()]
    fcs = []
    for fc in (ens.fc1, ens.fc2):
        ref = nn.Linear(fc.w.shape[1], fc.w.shape[2])
        ref.weight.data, ref.bias.data = fc.w[m].T.clone(), fc.b[m, 0].clone()
        fcs.append(ref)
    return nn.Sequential(*layers, nn.Flatten(), fcs[0], nn.BatchNorm1d(100), nn.ReLU(), fcs[1])


def stack_input(x, ph):
    """(n, L) magnitudes + (n, L) phases -> (n, 2, L)."""
    return torch.stack([x, ph], dim=1)


def amp_on(amp, device):
    return bool(amp) and str(device).startswith("cuda")


def autocast(device, amp):
    """fp16 mixed precision on the GPU when amp is on; a no-op otherwise (e.g. in the offline CPU tests)."""
    on = amp_on(amp, device)
    return torch.autocast("cuda" if on else "cpu", enabled=on)


@torch.no_grad()
def predict_proba(model, X, batch=512, amp=True):
    """Softmax outputs of every member: (M, n, N). X: (n, 2, L) on the model's device."""
    model.eval()
    out = []
    for i in range(0, len(X), batch):
        xb = X[i:i + batch].unsqueeze(0).expand(model.M, -1, -1, -1)
        with autocast(X.device.type, amp):
            logits = model(xb)
        out.append(F.softmax(logits.float(), -1))
    return torch.cat(out, 1)


def macro_f1(y, pred, n_classes):
    f1 = []
    for c in range(n_classes):
        tp = np.sum((pred == c) & (y == c))
        p = tp / max(np.sum(pred == c), 1)
        r = tp / max(np.sum(y == c), 1)
        f1.append(0.0 if p + r == 0 else 2 * p * r / (p + r))
    return float(np.mean(f1))


def fit_ensembles(train, val, test, n_classes, runs=15, ens=5, max_epochs=10, patience=3, batch=128,
                  lr=1e-4, seed=0, warp=None, device="cuda", log=print, tag="", amp=True, checks_per_epoch=4):
    """Train `runs` independent ensembles of `ens` CNNs and score each run on val and test.

    train/val/test: dicts of torch tensors x (n, L), ph (n, L), y (n,). Training batches are class-balanced
    (the resampling block with gamma = 1, one independent stream per member), one epoch = len(train) samples.
    Each run is scored on val and test 4 times per epoch; we keep the check with the lowest ensemble validation
    loss (early stopping) and report its test scores. Measured on Catalina: the validation loss is lowest
    after ~2 epochs and test accuracy then falls from 74% to 67% by epoch 10, because the balanced batches
    repeat the rarest class ~66 times per epoch. So training stops once no run improved for `patience` epochs.
    `warp` optionally augments every training batch (the soft-warping baselines).
    """
    torch.manual_seed(seed)
    M = runs * ens
    L = train["x"].shape[1]
    model = CNNEnsemble(M, n_classes, L).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, betas=(0.9, 0.999))
    scaler = torch.amp.GradScaler(enabled=amp_on(amp, device))
    Xtr = stack_input(train["x"], train["ph"]).to(device)
    ytr = train["y"].to(device)
    Xva, yva = stack_input(val["x"], val["ph"]).to(device), val["y"].cpu().numpy()
    Xte, yte = stack_input(test["x"], test["ph"]).to(device), test["y"].cpu().numpy()
    y_np = train["y"].cpu().numpy()
    samplers = [ClassResampler(y_np, gamma=1.0, seed=seed * 1000 + m, n_classes=n_classes) for m in range(M)]
    steps = math.ceil(len(y_np) / batch / checks_per_epoch)            # training steps between two checks

    best_loss = np.full(runs, np.inf)
    best = {"epoch": np.zeros(runs), "val_acc": np.zeros(runs), "test_acc": np.zeros(runs),
            "test_f1": np.zeros(runs), "test_pred": np.zeros((runs, len(yte)), int)}
    curves = {"epoch": [], "train_loss": [], "val_loss": [], "val_acc": [], "test_acc": []}
    since = np.zeros(runs, int)
    for check in range(1, int(max_epochs * checks_per_epoch) + 1):
        epoch = check / checks_per_epoch
        model.train()
        idx = torch.from_numpy(np.stack([s.sample(steps * batch) for s in samplers])).to(device)   # (M, S*B)
        tot = torch.zeros((), device=device)                            # summed on the GPU (no sync per step)
        for s in range(steps):
            ib = idx[:, s * batch:(s + 1) * batch]                    # (M, B)
            xb, yb = Xtr[ib], ytr[ib]                                  # (M, B, 2, L), (M, B)
            if warp is not None:
                xb = warp(xb)
            with autocast(device, amp):
                logits = model(xb)
            loss = F.cross_entropy(logits.float().reshape(-1, n_classes), yb.reshape(-1)) * M   # sum over members
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            tot += loss.detach() / M
        tot = tot.item()
        # scores of every run (= average of its members' softmax) on val and test: (runs, n, N)
        pv = predict_proba(model, Xva, amp=amp).reshape(runs, ens, len(yva), n_classes).mean(1).cpu().numpy()
        pt = predict_proba(model, Xte, amp=amp).reshape(runs, ens, len(yte), n_classes).mean(1).cpu().numpy()
        vloss = -np.log(np.take_along_axis(pv, yva[None, :, None], 2)[:, :, 0] + 1e-12).mean(1)   # (runs,)
        vacc = (pv.argmax(2) == yva[None]).mean(1)
        tpred = pt.argmax(2)                                                                    # (runs, n)
        tacc = (tpred == yte[None]).mean(1)
        for r in range(runs):
            if vloss[r] < best_loss[r]:
                best_loss[r], since[r] = vloss[r], 0
                best["epoch"][r], best["val_acc"][r], best["test_acc"][r] = epoch, vacc[r], tacc[r]
                best["test_f1"][r] = macro_f1(yte, tpred[r], n_classes)
                best["test_pred"][r] = tpred[r]
            else:
                since[r] += 1
        curves["epoch"].append(epoch)
        curves["train_loss"].append(tot / steps)
        curves["val_loss"].append(float(vloss.mean()))
        curves["val_acc"].append(float(vacc.mean()))
        curves["test_acc"].append(float(tacc.mean()))
        if check % checks_per_epoch == 0:
            log(f"  {tag} epoch {epoch:.0f}: train loss {tot / steps:.3f} | val loss {vloss.mean():.3f} "
                f"acc {vacc.mean():.3f} | test acc {tacc.mean():.3f}")
        if (since >= patience * checks_per_epoch).all():
            break
    recall = np.array([[np.mean(best["test_pred"][r][yte == c] == c) for c in range(n_classes)]
                       for r in range(runs)])
    return {"test_acc": best["test_acc"].tolist(), "test_f1": best["test_f1"].tolist(),
            "val_acc": best["val_acc"].tolist(), "best_epoch": best["epoch"].tolist(),
            "test_recall_per_class": recall.tolist(), "epochs_run": epoch, "curves": curves}


def train_feature_cnn(train, val, n_classes, max_epochs=20, patience=4, batch=128, lr=1e-4, seed=0,
                      device="cuda", log=print):
    """One base classifier trained on the balanced real training set (D_train^u). Its last conv block is the
    feature space for density, recall, FID and the G-score (Sec. 4.3). Keeps the lowest-val-loss weights.
    (Full precision: it is a single small network and its features are reused by every metric.)"""
    torch.manual_seed(seed)
    L = train["x"].shape[1]
    model = CNNEnsemble(1, n_classes, L).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, betas=(0.9, 0.999))
    Xtr, ytr = stack_input(train["x"], train["ph"]).to(device), train["y"].to(device)
    Xva, yva = stack_input(val["x"], val["ph"]).to(device), val["y"].to(device)
    sampler = ClassResampler(train["y"].cpu().numpy(), gamma=1.0, seed=seed, n_classes=n_classes)
    steps = math.ceil(len(ytr) / batch)
    best, best_state, since, curve = np.inf, None, 0, []
    for epoch in range(1, max_epochs + 1):
        model.train()
        idx = torch.from_numpy(sampler.sample(steps * batch)).to(device)
        tot = torch.zeros((), device=device)
        for s in range(steps):
            ib = idx[s * batch:(s + 1) * batch]
            loss = F.cross_entropy(model(Xtr[ib].unsqueeze(0))[0], ytr[ib])
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            tot += loss.detach()
        p = predict_proba(model, Xva, amp=False)[0]
        vloss = F.nll_loss(torch.log(p + 1e-12), yva).item()
        vacc = (p.argmax(1) == yva).float().mean().item()
        curve.append((tot.item() / steps, vloss, vacc))
        if vloss < best:
            best, since = vloss, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            since += 1
        if since >= patience:
            break
    model.load_state_dict(best_state)
    model.eval()
    log(f"feature CNN: {epoch} epochs, best val loss {best:.3f}, val acc at best "
        f"{[c[2] for c in curve][int(np.argmin([c[1] for c in curve]))]:.3f}")
    return model, curve


@torch.no_grad()
def extract_features(model, x, ph, batch=1024):
    """Feature vectors (n, 64 * ceil(L/32)) of light curves x, ph (torch tensors on the model's device)."""
    model.eval()
    out = []
    for i in range(0, len(x), batch):
        out.append(model.features(stack_input(x[i:i + batch], ph[i:i + batch]).unsqueeze(0))[0])
    return torch.cat(out)


In [ ]:
%%writefile astra/pelsvae.py
"""PELS-VAE data (Martinez-Palomera, Bloom & Abrahams 2020), loaded exactly as their released code does
(github.com/jorgemarpa/PELS-VAE, src/datasets.py and vae_main.py).

The Zenodo training file holds 65,760 OGLE-III I-band light curves of 8 classes: 33,124 real stars
(phase-folded, 300 random points) plus 32,636 augmented copies (magnitudes resampled from N(m, err),
random phase shift, re-sampled). Every curve is shown over 2 phase cycles, so it has 600 points.
Each curve is min-max scaled on its own to [0.0001, 0.9999] (phase and magnitude; errors by the
magnitude range). Their split: a random 80/20 train/validation split with seed 13.
"""
import gzip

import numpy as np
import pandas as pd

from .common import download

ZENODO = "https://zenodo.org/records/3820679/files/"
TRAIN_FILE = "OGLE3_lcs_I_meta_snr5_augmented_folded_trim600.npy.gz"
PHYS = {"P": "Period", "T": "teff_val", "M": "[Fe/H]_J95", "C": "bp_rp", "A": "abs_Gmag", "R": "radius_val",
        "L": "lum_val"}           # their --phy letters, in their order


def load_raw(data_dir):
    path = download(ZENODO + TRAIN_FILE + "?download=1", data_dir / TRAIN_FILE)
    with gzip.open(path, "rb") as f:
        d = np.load(f, allow_pickle=True).item()
    return d["lcs"].astype(np.float32), d["meta"].reset_index(drop=True)


def normalize_each(lcs, lo=0.0001, hi=0.9999):
    """Their utils.normalize_each(norm_time=True, scale_to=[.0001, .9999]), vectorised."""
    out = np.empty_like(lcs)
    t, m, e = lcs[..., 0], lcs[..., 1], lcs[..., 2]
    tr = t.max(1, keepdims=True) - t.min(1, keepdims=True)
    mr = m.max(1, keepdims=True) - m.min(1, keepdims=True)
    out[..., 0] = (t - t.min(1, keepdims=True)) / tr * (hi - lo) + lo
    out[..., 1] = (m - m.min(1, keepdims=True)) / mr * (hi - lo) + lo
    out[..., 2] = e / mr * (hi - lo)
    return out


def make_dataset(lcs_norm, meta, phy=""):
    """Their Astro_lightcurves(...) + remove_nan(): one-hot labels (alphabetical classes), min-max scaled
    physical parameters (scaler fitted on all rows, NaNs ignored), rows with missing parameters dropped."""
    classes = sorted(meta.Type.unique())
    y = meta.Type.map({c: i for i, c in enumerate(classes)}).values.astype(np.int64)
    names = [PHYS[k] for k in "PTMCARL" if k in phy.upper()]
    cols = names if names else ["Period"]
    p = meta[cols].values.astype(np.float32)
    lo, hi = np.nanmin(p, 0), np.nanmax(p, 0)
    p = (p - lo) / (hi - lo)
    keep = ~np.isnan(p).any(1)
    return {"x": lcs_norm[keep], "y": y[keep], "phy": p[keep] if names else np.zeros((keep.sum(), 0), np.float32),
            "rows": np.flatnonzero(keep), "classes": classes, "phy_names": names}


def split_indices(n, seed=13, test_split=0.2):
    """Their get_dataloader(): np.random.seed(seed); shuffle list(range(n)); first 20% = validation."""
    np.random.seed(seed)
    idx = list(range(n))
    np.random.shuffle(idx)
    k = int(np.floor(test_split * n))
    return np.array(idx[k:]), np.array(idx[:k])


def class_counts(meta):
    real = meta.drop_duplicates("OGLE_id")
    return pd.DataFrame({"curves (with augmented)": meta.Type.value_counts().sort_index(),
                         "unique stars": real.Type.value_counts().sort_index()})


In [ ]:
%%writefile astra/bench_data.py
"""Data for the ASTRA benchmark: the pre-processed light curves of the GAN notebook, split over stars.

What stays as in Garcia-Jara et al. (2022): the curves (L magnitudes at L sorted phases, median-centred), and
what a generator is given, the condition (phases, amplitude, class) of a real star.

What changes for the benchmark (ours):
  - the split. The paper holds out 20% of the smallest class for every class (36 curves per class on Catalina),
    and on ZTF it splits over curves, so the g and the r curve of a star can sit on both sides. Here the split
    is over STARS and stratified by class: 70% train, 10% validation, 20% test. Curves that share class and
    period are treated as one star (ZTF gives a star seen in two fields two object ids, with the same
    catalogue period), so no star is on two sides;
  - the target. Every model generates the amplitude-normalised curve u = magnitude / amplitude, which lies in
    [-2, 2] for every class, and is told the amplitude as log10(amplitude), standardised. A generated curve is
    turned back into magnitudes by multiplying with the amplitude of its condition. rep = "raw" gives the
    paper's own target instead (magnitudes, amplitude in mag), for one control run of the GAN.
"""
import hashlib

import numpy as np
import torch
import torch.nn.functional as F

from .sampling import ClassResampler

TRAIN, VAL, TEST = 0, 1, 2
FRACTIONS = (0.7, 0.1, 0.2)          # train, validation, test (of the stars of every class)
RARE_BELOW = 1000                    # a class with fewer training curves than this is in the "rare tier"
LATENT = {100: 16, 40: 8}            # noise / latent size per curve length: Catalina, ZTF (Sec. 4.2 of the paper)
G_PARAMS = {100: 3_990_785, 40: 3_708_161}   # parameters of the paper's generator; the other families match it


def star_ids(y, period):
    """One id per (class, period): the curves of one star, whatever band or object id they come from."""
    key = np.stack([y.astype(np.float64), period.astype(np.float64)], 1)
    return np.unique(key, axis=0, return_inverse=True)[1].reshape(-1)


def star_split(y, star, seed, fractions=FRACTIONS):
    """Stratified split over stars: every class keeps the same train/val/test shares, a star is on one side."""
    rng = np.random.default_rng(1000 + seed)
    stars, first = np.unique(star, return_index=True)
    ys = y[first]
    side = np.full(len(stars), TRAIN, np.int8)
    for c in np.unique(ys):
        idx = rng.permutation(np.flatnonzero(ys == c))
        n_val, n_test = int(round(fractions[1] * len(idx))), int(round(fractions[2] * len(idx)))
        side[idx[:n_val]] = VAL
        side[idx[n_val:n_val + n_test]] = TEST
    return side[np.searchsorted(stars, star)]


def split_table(npz_path, seed):
    """Curves per class and side, for the notebook's data section."""
    z = np.load(npz_path)
    split = star_split(z["y"], star_ids(z["y"], z["period"]), seed)
    n = len(z["classes"])
    return {name: np.bincount(z["y"][split == s], minlength=n) for name, s in (("train", TRAIN), ("val", VAL),
                                                                               ("test", TEST))}


def load(npz_path, seed, device, rep="unit"):
    """Everything a job needs, as tensors on `device`. Index arrays always refer to rows of the file."""
    z = np.load(npz_path)
    y = z["y"].astype(np.int64)
    star = star_ids(y, z["period"])
    split = star_split(y, star, seed)
    n_classes = len(z["classes"])
    t = lambda a, dt=torch.float32: torch.from_numpy(np.ascontiguousarray(a)).to(device=device, dtype=dt)
    data = {"x": t(z["mag"]), "ph": t(z["phase"]), "amp": t(z["amp"]), "y": t(y, torch.long),
            "n_classes": n_classes, "classes": [str(c) for c in z["classes"]], "L": z["mag"].shape[1],
            "y_np": y, "split": split, "star": star, "seed": seed, "rep": rep, "device": device}
    data["onehot"] = F.one_hot(data["y"], n_classes).float()
    for name, s in (("train", TRAIN), ("val", VAL), ("test", TEST)):
        idx = np.flatnonzero(split == s)
        data[f"{name}_idx_np"] = idx
        data[f"{name}_idx"] = torch.from_numpy(idx).to(device)
    tr = data["train_idx"]
    assert (data["amp"] > 0).all(), "every curve needs a positive amplitude"
    if rep == "unit":                 # u = x / amplitude in [-2, 2]; condition = standardised log-amplitude
        la = torch.log10(data["amp"])
        data["u"] = data["x"] / data["amp"][:, None]
        data["camp"] = (la - la[tr].mean()) / la[tr].std()
        data["mag_scale"] = data["amp"].clone()
        data["out_scale"] = 2.0
    elif rep == "raw":                # the paper's target: magnitudes, amplitude in mag, tanh scale = largest |mag|
        data["u"], data["camp"] = data["x"], data["amp"]
        data["mag_scale"] = torch.ones_like(data["amp"])
        data["out_scale"] = float(data["x"][tr].abs().max())
    else:
        raise ValueError(rep)
    counts = np.bincount(y[split == TRAIN], minlength=n_classes)
    data["train_counts"] = counts
    data["rare"] = counts < RARE_BELOW
    h = hashlib.md5()
    for a in (y, split, z["mag"], z["phase"]):
        h.update(np.ascontiguousarray(a).tobytes())
    data["hash"] = h.hexdigest()[:12]            # two notebooks may pair their results only if this agrees
    return data


def balanced_idx(data, n, seed, gamma=1.0, side="train"):
    """n row indices of one side, drawn by the resampling block (gamma = 1: every class equally often)."""
    pool = data[f"{side}_idx_np"]
    rs = ClassResampler(data["y_np"][pool], gamma, seed=seed, n_classes=data["n_classes"])
    return data[f"{side}_idx"][torch.from_numpy(rs.sample(n)).to(data["device"])]


def subset(data, idx, x=None):
    """A set of curves as the classifier wants it: magnitudes x, phases ph, labels y."""
    return {"x": data["x"][idx] if x is None else x, "ph": data["ph"][idx], "y": data["y"][idx]}


In [ ]:
%%writefile astra/data_audit.py
"""Dataset audit for the ASTRA project: tables and figures that describe the light-curve datasets and justify
the choices of the benchmark (split over stars, balanced accuracy, the rare tier, the amplitude-normalised target).

Everything here runs on a CPU in a few minutes and trains nothing except one small classifier per dataset
(for the confusion matrix and the feature map). Z is a pre-processed file (catalina_L100.npz or ztf_L40.npz)
opened with numpy.load; its `split` column is the split of Garcia-Jara et al. (2022), kept for comparison.
"""
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from . import bench_data, classifier

TRAIN, VAL, TEST = 0, 1, 2
COLORS = plt.get_cmap("tab10").colors


def classes_of(Z):
    return [str(c) for c in Z["classes"]]


def object_ids(Z):
    return Z["numid"] if "numid" in Z.files else Z["oid"]


def class_table(Z, seed=0):
    """Curves, object ids and stars per class, and the sizes of our split over stars and of the paper's split."""
    y, names = Z["y"], classes_of(Z)
    star = bench_data.star_ids(y, Z["period"])
    ours = bench_data.star_split(y, star, seed)
    rows = []
    for c, name in enumerate(names):
        m = y == c
        rows.append({"class": name, "curves": int(m.sum()), "object ids": len(np.unique(object_ids(Z)[m])),
                     "stars": len(np.unique(star[m])),
                     "train": int((m & (ours == TRAIN)).sum()), "val": int((m & (ours == VAL)).sum()),
                     "test": int((m & (ours == TEST)).sum()),
                     "paper train": int((m & (Z["split"] == TRAIN)).sum()), "paper test": int((m & (Z["split"] == TEST)).sum())})
    t = pd.DataFrame(rows)
    t["share of curves %"] = (100 * t.curves / t.curves.sum()).round(2)
    t["rare tier"] = t.train < bench_data.RARE_BELOW
    return t


def plot_imbalance(tables, path):
    """Curves per class on a log scale, one panel per dataset; the rare tier is hatched."""
    fig, ax = plt.subplots(1, len(tables), figsize=(6 * len(tables), 3.6), squeeze=False)
    for a, (name, t) in zip(ax[0], tables.items()):
        bars = a.bar(t["class"], t.curves, color=[COLORS[i] for i in range(len(t))])
        for b, rare, n in zip(bars, t["rare tier"], t.curves):
            if rare:
                b.set_hatch("//")
            a.text(b.get_x() + b.get_width() / 2, n * 1.08, f"{n:,}", ha="center", fontsize=8)
        a.set_yscale("log"); a.set_ylabel("light curves (log scale)")
        a.set_title(f"{name}: {t.curves.sum():,} curves, largest / smallest class = {t.curves.max() / t.curves.min():.0f}x "
                    "(hatched = rare tier)", fontsize=9)
    fig.tight_layout(); fig.savefig(path, dpi=130); plt.show()


def plot_gallery(Z, name, path, per_class=3, seed=1):
    """Folded light curves, a few per class, as the models see them (magnitudes grow downwards)."""
    names, rng = classes_of(Z), np.random.default_rng(seed)
    fig, ax = plt.subplots(per_class, len(names), figsize=(2.9 * len(names), 2.1 * per_class), sharex=True, squeeze=False)
    for c, cname in enumerate(names):
        for r, i in enumerate(rng.choice(np.flatnonzero(Z["y"] == c), per_class, replace=False)):
            err = np.where((Z["err"][i] > 0) & (Z["err"][i] < 1), Z["err"][i], 0)
            ax[r, c].errorbar(Z["phase"][i], Z["mag"][i], yerr=err, fmt=".", ms=3, lw=0.5, color=COLORS[c], ecolor="0.75")
            ax[r, c].invert_yaxis()
            ax[r, c].set_title(f"{cname} · P = {Z['period'][i]:.3g} d · A = {Z['amp'][i]:.2f}", fontsize=8)
    fig.supxlabel("phase"); fig.supylabel("centred magnitude")
    fig.suptitle(f"{name}: folded light curves ({Z['mag'].shape[1]} points each)", fontsize=10)
    fig.tight_layout(); fig.savefig(path, dpi=130); plt.show()


def plot_period_amplitude(Z, name, path, max_per_class=3000, seed=0):
    """Period against amplitude: what the conditions (class, amplitude) and the period say about a star."""
    names, rng = classes_of(Z), np.random.default_rng(seed)
    fig, ax = plt.subplots(1, 3, figsize=(16, 4.2), gridspec_kw={"width_ratios": [1.5, 1, 1]})
    order = np.argsort(-np.bincount(Z["y"], minlength=len(names)))          # common classes first, rare ones on top
    for c in order:
        i = np.flatnonzero(Z["y"] == c)
        i = rng.choice(i, min(len(i), max_per_class), replace=False)
        ax[0].scatter(Z["period"][i], Z["amp"][i], s=4, alpha=0.35, color=COLORS[c], label=names[c])
    ax[0].set_xscale("log"); ax[0].set_yscale("log"); ax[0].set_xlabel("period (days)"); ax[0].set_ylabel("amplitude (mag)")
    leg = ax[0].legend(fontsize=8, markerscale=3)
    for h in getattr(leg, "legend_handles", getattr(leg, "legendHandles", [])):
        h.set_alpha(1)
    for a, key, label in ((ax[1], "period", "log10 period (days)"), (ax[2], "amp", "log10 amplitude (mag)")):
        parts = a.violinplot([np.log10(Z[key][Z["y"] == c]) for c in range(len(names))], showmedians=True, widths=0.9)
        for body, c in zip(parts["bodies"], range(len(names))):
            body.set_facecolor(COLORS[c]); body.set_alpha(0.6)
        a.set_xticks(range(1, len(names) + 1)); a.set_xticklabels(names, fontsize=8); a.set_ylabel(label)
    fig.suptitle(f"{name}: period and amplitude per class", fontsize=10)
    fig.tight_layout(); fig.savefig(path, dpi=130); plt.show()
    return pd.DataFrame({"class": names,
                         "median period (d)": [float(np.median(Z["period"][Z["y"] == c])) for c in range(len(names))],
                         "median amplitude (mag)": [float(np.median(Z["amp"][Z["y"] == c])) for c in range(len(names))]}).round(3)


def largest_gap(phase):
    """Largest empty stretch of the phase circle for every curve (phases sorted in [0, 1))."""
    gaps = np.diff(phase, axis=1)
    wrap = 1 - phase[:, -1] + phase[:, 0]
    return np.maximum(gaps.max(1), wrap)


def plot_sampling(Z, name, path, seed=0):
    """How irregular the sampling is: the largest phase gap of every curve, against what L uniformly random
    phases would give."""
    names, L = classes_of(Z), Z["phase"].shape[1]
    gap = largest_gap(Z["phase"])
    ref = np.median(largest_gap(np.sort(np.random.default_rng(seed).random((20_000, L)), 1)))
    fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
    bins = np.linspace(0, 0.6, 61)
    for c, cname in enumerate(names):
        ax[0].hist(gap[Z["y"] == c], bins=bins, histtype="step", density=True, color=COLORS[c], label=cname)
    ax[0].axvline(ref, color="k", ls="--", lw=1, label=f"{L} random phases (median)")
    ax[0].set_xlabel("largest phase gap of a curve"); ax[0].set_ylabel("density"); ax[0].legend(fontsize=7)
    worst = np.argsort(gap)[-3:]
    for k, i in enumerate(worst):
        ax[1].plot(Z["phase"][i], Z["mag"][i] - Z["mag"][i].mean() + 1.2 * k * Z["amp"][i].clip(0.1), ".", ms=4,
                   label=f"{names[Z['y'][i]]}, gap {gap[i]:.2f}")
    ax[1].invert_yaxis(); ax[1].set_xlabel("phase"); ax[1].set_yticks([]); ax[1].legend(fontsize=7)
    ax[1].set_title("the three curves with the largest gaps (shifted vertically)", fontsize=9)
    fig.suptitle(f"{name}: irregular sampling in phase", fontsize=10)
    fig.tight_layout(); fig.savefig(path, dpi=130); plt.show()
    t = pd.DataFrame({"class": names,
                      "median largest gap": [float(np.median(gap[Z["y"] == c])) for c in range(len(names))],
                      "curves with a gap > 0.2 (%)": [100 * float(np.mean(gap[Z["y"] == c] > 0.2)) for c in range(len(names))]})
    t["random phases would give"] = ref
    return t.round(3)


def plot_noise(Z, name, path):
    """Photometric error relative to the amplitude: how much of a curve is signal."""
    names = classes_of(Z)
    err = np.where((Z["err"] > 0) & (Z["err"] < 1), Z["err"], np.nan)            # flagged errors (>= 1 mag) ignored
    med = np.nanmedian(err, 1)
    rel = med / Z["amp"]
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
    for a, v, label in ((ax[0], med, "median error of a curve (mag)"), (ax[1], rel, "median error / amplitude")):
        data = [np.log10(v[(Z["y"] == c) & np.isfinite(v)]) for c in range(len(names))]
        parts = a.violinplot(data, showmedians=True, widths=0.9)
        for body, c in zip(parts["bodies"], range(len(names))):
            body.set_facecolor(COLORS[c]); body.set_alpha(0.6)
        a.set_xticks(range(1, len(names) + 1)); a.set_xticklabels(names, fontsize=8); a.set_ylabel("log10 " + label)
    fig.suptitle(f"{name}: photometric noise per class", fontsize=10)
    fig.tight_layout(); fig.savefig(path, dpi=130); plt.show()
    return pd.DataFrame({"class": names,
                         "median error (mag)": [float(np.nanmedian(med[Z["y"] == c])) for c in range(len(names))],
                         "median error / amplitude": [float(np.nanmedian(rel[Z["y"] == c])) for c in range(len(names))]}).round(3)


def leakage_table(Z, seed=0):
    """Do curves of one star sit on both sides of a split? A star = all curves with the same class and period
    (two bands, or two object ids of one catalogue source). Paper's split: the one stored in the file."""
    y, names = Z["y"], classes_of(Z)
    star = bench_data.star_ids(y, Z["period"])
    ours = bench_data.star_split(y, star, seed)
    per_star = np.bincount(star)
    rows = []
    for c, name in enumerate(names):
        m = y == c
        row = {"class": name, "curves": int(m.sum()), "stars": len(np.unique(star[m])),
               "curves in stars with more than one curve (%)": 100 * float(np.mean(per_star[star[m]] > 1))}
        for label, split in (("paper's split", Z["split"]), ("our split", ours)):
            train_stars = np.unique(star[split == TRAIN])
            held = m & (split != TRAIN)
            row[f"{label}: held-out curves with a same-star curve in training (%)"] = \
                100 * float(np.isin(star[held], train_stars).mean()) if held.any() else np.nan
        rows.append(row)
    return pd.DataFrame(rows).round(1)


def power_table(Z, seed=0, recall=0.75):
    """How finely a test set can measure a class: curves per class, what one curve is worth in recall points,
    and the binomial standard error of a recall of 75%."""
    y, names = Z["y"], classes_of(Z)
    ours = bench_data.star_split(y, bench_data.star_ids(y, Z["period"]), seed)
    rows = []
    for c, name in enumerate(names):
        row = {"class": name}
        for label, split in (("paper", Z["split"]), ("ours", ours)):
            n = int(((y == c) & (split == TEST)).sum())
            row[f"test curves ({label})"] = n
            row[f"one curve = points ({label})"] = 100 / max(n, 1)
            row[f"std. error of a 75% recall ({label})"] = 100 * np.sqrt(recall * (1 - recall) / max(n, 1))
        rows.append(row)
    return pd.DataFrame(rows).round(2)


def confusion_and_map(npz_path, name, path, seed=0, max_epochs=6, n_map=400, device="cpu"):
    """One CNN (the paper's base classifier) trained on the real training curves of our split: its confusion
    matrix on the test stars, and a t-SNE map of its last-conv features for a class-balanced sample of them."""
    from sklearn.manifold import TSNE
    data = bench_data.load(npz_path, seed, device)
    names, N = data["classes"], data["n_classes"]
    tr, va, te = (bench_data.subset(data, data[f"{s}_idx"]) for s in ("train", "val", "test"))
    model, _ = classifier.train_feature_cnn(tr, va, N, max_epochs=max_epochs, seed=seed, device=device, log=lambda *a: None)
    X = classifier.stack_input(te["x"], te["ph"])
    pred = classifier.predict_proba(model, X, amp=False)[0].argmax(1).cpu().numpy()
    yte = data["y_np"][data["test_idx_np"]]
    conf = np.zeros((N, N))
    np.add.at(conf, (yte, pred), 1)
    rec = conf / conf.sum(1, keepdims=True)
    rng = np.random.default_rng(seed)
    pick = np.concatenate([rng.choice(np.flatnonzero(yte == c), min(n_map, int((yte == c).sum())), replace=False)
                           for c in range(N)])
    feats = classifier.extract_features(model, te["x"][torch.from_numpy(pick)], te["ph"][torch.from_numpy(pick)]).cpu().numpy()
    xy = TSNE(n_components=2, init="pca", perplexity=30, random_state=seed).fit_transform(feats)
    fig, ax = plt.subplots(1, 2, figsize=(13, 5))
    im = ax[0].imshow(100 * rec, cmap="Blues", vmin=0, vmax=100)
    for i in range(N):
        for j in range(N):
            ax[0].text(j, i, f"{100 * rec[i, j]:.0f}", ha="center", va="center", fontsize=9,
                       color="w" if rec[i, j] > 0.5 else "k")
    ax[0].set_xticks(range(N)); ax[0].set_xticklabels(names, fontsize=8); ax[0].set_yticks(range(N))
    ax[0].set_yticklabels([f"{n} ({int(conf[i].sum())})" for i, n in enumerate(names)], fontsize=8)
    ax[0].set_xlabel("predicted class"); ax[0].set_ylabel("true class (test curves)")
    ax[0].set_title(f"confusion matrix, % of each true class | balanced accuracy {100 * np.mean(np.diag(rec)):.1f}%, "
                    f"accuracy {100 * np.mean(pred == yte):.1f}%", fontsize=9)
    plt.colorbar(im, ax=ax[0], fraction=0.046)
    for c in np.argsort(-np.bincount(yte[pick], minlength=N)):
        m = yte[pick] == c
        ax[1].scatter(xy[m, 0], xy[m, 1], s=7, alpha=0.6, color=COLORS[c], label=names[c])
    ax[1].legend(fontsize=8, markerscale=2); ax[1].set_xticks([]); ax[1].set_yticks([])
    ax[1].set_title(f"t-SNE of the classifier's features (up to {n_map} test curves per class)", fontsize=9)
    fig.suptitle(f"{name}: where a classifier trained on real data confuses the classes (one CNN, split over stars)", fontsize=10)
    fig.tight_layout(); fig.savefig(path, dpi=130); plt.show()
    return pd.DataFrame(100 * rec, index=names, columns=names).round(1), \
        {"balanced accuracy %": 100 * float(np.mean(np.diag(rec))), "accuracy %": 100 * float(np.mean(pred == yte))}


def ogle_table(meta, split_seed=13):
    """The PELS-VAE training file: curves against unique stars per class, the share of curves that have the
    three physical parameters, and how many validation stars of their random split have a copy in training."""
    from .pelsvae import split_indices
    tr, va = split_indices(len(meta), split_seed)
    ids, phys = meta.OGLE_id.values, meta[["Period", "teff_val", "abs_Gmag"]].notna().all(1).values
    in_train = set(ids[tr])
    rows = []
    for name in sorted(meta.Type.unique()):
        m = meta.Type.values == name
        stars_va = set(ids[va][m[va]])
        rows.append({"class": name, "curves": int(m.sum()), "stars": len(set(ids[m])),
                     "curves per star": m.sum() / len(set(ids[m])),
                     "curves with period, T_eff and abs. G mag (%)": 100 * float(phys[m].mean()),
                     "validation stars with a copy in training (%)": 100 * len(stars_va & in_train) / max(len(stars_va), 1)})
    return pd.DataFrame(rows).round(1)


def plot_ogle(t, path):
    fig, ax = plt.subplots(figsize=(8, 3.6))
    x = np.arange(len(t))
    ax.bar(x - 0.2, t.stars, 0.4, label="unique stars"); ax.bar(x + 0.2, t.curves, 0.4, label="curves in the training file")
    for i, r in t.iterrows():
        ax.text(i + 0.2, r.curves * 1.1, f"x{r['curves per star']:.0f}", ha="center", fontsize=8)
    ax.set_xticks(x); ax.set_xticklabels(t["class"], fontsize=8); ax.set_yscale("log"); ax.legend(fontsize=8)
    ax.set_title("PELS-VAE training file (OGLE-III): every class is filled up with augmented copies", fontsize=9)
    fig.tight_layout(); fig.savefig(path, dpi=130); plt.show()


def pipeline_example(catalog_path, phot_path, Z, path, seed=0):
    """Catalina only: two stars from raw photometry to model input
    (time series -> folded by its period -> outlier filter -> L random points, median-centred)."""
    from . import catalina
    cat = catalina.read_catalog(catalog_path).set_index("numid")
    rng = np.random.default_rng(seed)
    picks = [int(Z["numid"][rng.choice(np.flatnonzero(Z["y"] == c))]) for c in (1, len(Z["classes"]) - 1)]   # an RRL and a CEP
    phot = catalina.read_photometry(phot_path, picks)
    fig, ax = plt.subplots(len(picks), 3, figsize=(15, 3.2 * len(picks)), squeeze=False)
    for r, numid in enumerate(picks):
        t, m, e = phot[numid]
        period = float(cat.loc[numid, "period"])
        ph = catalina.fold(t, period)
        o = np.argsort(ph, kind="stable")
        keep = catalina.zscore_keep(m[o])
        i = int(np.flatnonzero(Z["numid"] == numid)[0])
        ax[r, 0].plot(t - t.min(), m, ".", ms=3, color="0.3"); ax[r, 0].invert_yaxis()
        ax[r, 0].set_xlabel("days since the first observation"); ax[r, 0].set_ylabel("magnitude")
        ax[r, 0].set_title(f"{cat.loc[numid, 'cls']}: {len(t)} observations over {(t.max() - t.min()) / 365.25:.1f} years", fontsize=9)
        ax[r, 1].plot(ph[o][keep], m[o][keep], ".", ms=3, color="C0", label="kept")
        ax[r, 1].plot(ph[o][~keep], m[o][~keep], "x", ms=5, color="C3", label="outliers removed")
        ax[r, 1].invert_yaxis(); ax[r, 1].set_xlabel("phase = (time mod period) / period"); ax[r, 1].legend(fontsize=7)
        ax[r, 1].set_title(f"folded with its period, {period:.4g} days", fontsize=9)
        ax[r, 2].plot(Z["phase"][i], Z["mag"][i], ".", ms=4, color="C1"); ax[r, 2].invert_yaxis()
        ax[r, 2].set_xlabel("phase"); ax[r, 2].set_title(f"model input: {Z['mag'].shape[1]} random points, median-centred", fontsize=9)
    fig.suptitle("Catalina: from raw photometry to the curve a model sees", fontsize=10)
    fig.tight_layout(); fig.savefig(path, dpi=130); plt.show()


## 1 · The datasets

Both files hold folded, median-centred light curves with a fixed number of points, as the models see them. A *star* is counted as all curves with the same class and period (in ZTF one star can have a g curve and an r curve, and two object ids).

In [ ]:
# ---- 1 · Get the pre-processed files and summarise them ----
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from astra import catalina, bench_data, data_audit as audit
from astra.common import download, find_input

pd.set_option("display.width", 250)
Z_ALL, NPZ = {}, {}
for name in DATASETS:
    path = WORK / "data" / f"{name}_L{L[name]}.npz"
    if not path.exists():
        ready = find_input(path.name)
        if ready:
            shutil.copy2(ready, path)
        elif name == "catalina":                         # download + pre-process, about 3 minutes
            out, _ = catalina.preprocess(WORK / "data", L=L[name], class_range_k=CLASS_RANGE_K, seed=DATA_SEED)
            np.savez_compressed(path, **out)
        else:
            print(f"({path.name} is not attached: the {name} parts are skipped)")
            continue
    Z_ALL[name], NPZ[name] = np.load(path), path

rows = []
for name, Z in Z_ALL.items():
    t = audit.class_table(Z, SEED)
    rows.append({"dataset": name, "curves": len(Z["y"]), "stars": int(t.stars.sum()), "classes": len(t),
                 "points per curve": Z["mag"].shape[1], "largest class": f"{t['class'][t.curves.idxmax()]} ({t.curves.max():,})",
                 "smallest class": f"{t['class'][t.curves.idxmin()]} ({t.curves.min():,})",
                 "largest / smallest": round(t.curves.max() / t.curves.min()),
                 "rare tier (< 1,000 training curves)": ", ".join(t["class"][t["rare tier"]]),
                 "median amplitude (mag)": round(float(np.median(Z["amp"])), 3),
                 "median period (d)": round(float(np.median(Z["period"])), 3)})
show("The datasets", pd.DataFrame(rows), "datasets_summary.csv")

## 2 · Classes

- **Imbalance.** Curves per class on a log scale. The hatched classes have fewer than 1,000 training curves in the benchmark's split: the *rare tier*.
- **Gallery.** Three curves per class with their error bars. Magnitudes grow downwards, so a dip in brightness points down.
- **Pipeline** (Catalina). Two stars from the raw time series to the model input: observations spread over years, folded with the star's period, outliers removed, 100 random points kept and centred on their median.

In [ ]:
# ---- 2 · Classes: imbalance, example curves, and what pre-processing does ----
tables = {name: audit.class_table(Z, SEED) for name, Z in Z_ALL.items()}
for name, t in tables.items():
    show(f"{name}: curves per class, and the two splits (ours = over stars, split {SEED}; paper = stored in the file)",
         t, f"{name}_classes.csv")
safely("imbalance figure", lambda: audit.plot_imbalance(tables, FIGS / "class_imbalance.png"))
for name, Z in Z_ALL.items():
    safely(f"{name} gallery", lambda: audit.plot_gallery(Z, name, FIGS / f"{name}_gallery.png"))

if WITH_PIPELINE and "catalina" in Z_ALL:
    def pipeline():
        cat = download(catalina.CATALOG_URL, WORK / "data" / "CatalinaVars.tbl")
        phot = download(catalina.PHOT_URL, WORK / "data" / "AllVar.phot.gz")
        audit.pipeline_example(cat, phot, Z_ALL["catalina"], FIGS / "catalina_pipeline.png")
    safely("pipeline figure", pipeline)

## 3 · What the models are given, and what they have to produce

A model receives a real star's phases, amplitude and class, and generates the magnitudes.
- **Period and amplitude.** The left panel is the classic way to tell variable stars apart. The period itself is not given to the models: it is used up by the folding.
- **Sampling in phase.** The largest empty stretch of the phase circle for every curve, against what the same number of uniformly random phases would give. A gap much larger than that means the star's period is close to a multiple of the observing cadence.
- **Noise.** The median photometric error of a curve, and the same divided by the curve's amplitude: how much of the curve is signal.
- **Scale.** Raw magnitudes differ in scale by class; divided by the amplitude they do not. This is why the benchmark's target is magnitude / amplitude.

In [ ]:
# ---- 3 · Period, amplitude, sampling in phase, noise ----
for name, Z in Z_ALL.items():
    show(f"{name}: period and amplitude per class",
         safely(f"{name} period/amplitude", lambda: audit.plot_period_amplitude(Z, name, FIGS / f"{name}_period_amplitude.png")),
         f"{name}_period_amplitude.csv")
    show(f"{name}: largest phase gap per curve",
         safely(f"{name} sampling", lambda: audit.plot_sampling(Z, name, FIGS / f"{name}_sampling.png")),
         f"{name}_sampling.csv")
    show(f"{name}: photometric error per class",
         safely(f"{name} noise", lambda: audit.plot_noise(Z, name, FIGS / f"{name}_noise.png")),
         f"{name}_noise.csv")
    u = Z["mag"] / Z["amp"][:, None]
    print(f"\n{name}: magnitudes span {Z['mag'].min():.2f} to {Z['mag'].max():.2f} mag (std {Z['mag'].std():.3f}); "
          f"divided by each curve's amplitude they span {u.min():.2f} to {u.max():.2f} (std {u.std():.3f}) for every class")

## 4 · Splits

- **Leakage.** If two curves of one star sit on both sides of a split, the test is easier than it looks. The paper splits over curves; the benchmark splits over stars.
- **Resolution.** The paper tests on 20% of the smallest class for every class. The table gives, per class, the number of test curves, what one curve is worth in recall points, and the standard error of a recall of 75%.

In [ ]:
# ---- 4 · Splits: curves of one star on both sides, and how finely a test set measures a class ----
for name, Z in Z_ALL.items():
    show(f"{name}: stars with several curves, and held-out curves whose star is also in training",
         safely(f"{name} leakage", lambda: audit.leakage_table(Z, SEED)), f"{name}_leakage.csv")
    show(f"{name}: test-set size per class and what it can resolve (paper's split against ours)",
         safely(f"{name} power", lambda: audit.power_table(Z, SEED)), f"{name}_test_power.csv")

## 5 · Which classes are confused?

One CNN (the paper's base classifier) is trained on the real training curves with class-balanced batches and scored on the test stars of the benchmark's split. Rows of the matrix are true classes; a row shows where that class's curves go. The map on the right is a t-SNE of the classifier's features: classes that overlap there are the ones it cannot separate. It is one small network, not the 15 × 5 ensemble of the benchmark, so its numbers are only a guide.

In [ ]:
# ---- 5 · Which classes does a classifier trained on real data confuse? (one small CNN per dataset, CPU) ----
for name in Z_ALL:
    def run():
        conf, acc = audit.confusion_and_map(NPZ[name], name, FIGS / f"{name}_confusion.png", seed=SEED)
        show(f"{name}: confusion matrix of one CNN on the test stars, % of each true class (rows)", conf,
             f"{name}_confusion.csv", True)
        print({k: round(v, 1) for k, v in acc.items()})
    safely(f"{name} confusion", run)

## 6 · The PELS-VAE training file (OGLE-III)

Notebook 02 trains on the authors' own file. Every class in it is filled up to 5,000–10,000 curves with augmented copies of the real stars, and their split is random over curves. The table counts stars against curves, and the share of validation stars that have a copy in the training set.

In [ ]:
# ---- 6 · The PELS-VAE training file (OGLE-III): stars against curves ----
if WITH_OGLE:
    def ogle():
        from astra import pelsvae
        _, meta = pelsvae.load_raw(WORK / "data")
        t = audit.ogle_table(meta)
        show("PELS-VAE training file: curves, stars, parameters, and copies across their random split", t, "ogle_classes.csv")
        print(f"\nall classes: {len(meta):,} curves of {meta.OGLE_id.nunique():,} stars")
        audit.plot_ogle(t, FIGS / "ogle_stars_vs_curves.png")
    safely("OGLE section", ogle)

In [ ]:
# ---- Deliverables check ----
need = [TABLES / "datasets_summary.csv", FIGS / "class_imbalance.png"]
for name in Z_ALL:
    need += [FIGS / f"{name}_{f}.png" for f in ("gallery", "period_amplitude", "sampling", "noise", "confusion")]
    need += [TABLES / f"{name}_{f}.csv" for f in ("classes", "leakage", "test_power", "confusion")]
if WITH_PIPELINE and "catalina" in Z_ALL:
    need.append(FIGS / "catalina_pipeline.png")
if WITH_OGLE:
    need += [FIGS / "ogle_stars_vs_curves.png", TABLES / "ogle_classes.csv"]
missing = [str(p.relative_to(WORK)) for p in need if not p.exists()]
print(f"datasets audited: {list(Z_ALL)} | {len(list(FIGS.glob('*.png')))} figures, {len(list(TABLES.glob('*.csv')))} tables "
      f"| {(time.time() - RUN_STARTED) / 60:.1f} min")
print("ALL DELIVERABLES PRESENT" if not missing else f"MISSING: {missing}")

## 7 · Observations

From the run of 6 October 2026 (CPU, both files as used by notebook 01; split 0 of the benchmark). All numbers are in `tables_data/`.

**The datasets.** Catalina: 43,417 curves, 6 classes, 100 points per curve. ZTF: 56,135 curves of 30,125 stars, 5 classes, 40 points per curve. Both are dominated by eclipsing binaries: the largest class is 166 × the smallest on Catalina and 66 × on ZTF.

**Rare classes.** After a 70/10/20 split the rare tier has 347 (LPV), 172 (DSCT) and 127 (CEP) training curves on Catalina, and 365 (DSCT) and 329 (CEP) on ZTF. Together they are 2.1% of Catalina and 1.7% of ZTF.

**Period and amplitude.** The period alone almost separates the classes: DSCT near 0.09 days, the binaries and RR Lyrae between 0.3 and 0.9 days, Cepheids at 2–4 days, LPVs at 200–300 days. Amplitudes overlap (medians 0.24–0.42 mag) except for the LPVs (0.9 mag on Catalina, 1.7 on ZTF). The paper's classifier sees magnitudes and phases only; the period is used up by the folding. So a class that is easy to tell by its period can still be hard for this classifier, and the benchmark, which follows the paper, measures augmentation under that handicap. This belongs in the report's limitations.

**Scale.** Magnitudes span −3.7 to 4.4 mag on Catalina and −6.7 to 12.4 on ZTF; divided by each curve's amplitude they lie in [−2, 2] for every class. This is why the benchmark's target is magnitude / amplitude.

**Sampling in phase.** Catalina is close to what random phases would give: the median largest gap is 0.05–0.08 of a cycle against 0.05 for 100 random phases, and under 2% of the curves have a gap above 0.2. ZTF is much less regular: the median largest gap is 0.13–0.20 against 0.10 for 40 random phases, and a gap above 0.2 appears in 8% of the EB curves, 28% of RRL and DSCT, 17% of CEP and 50% of the LPVs. If telling a model the phases matters, it should show on ZTF; the phase ablation (`fm_nophase_s0`, notebook 06) should be read there first.

**Noise.** Catalina is noisy: the median photometric error is 0.19–0.27 of the amplitude for every class except the LPVs (0.06). ZTF is clean: 0.03–0.04. A generated Catalina curve without noise would not look like a real one, which is why the benchmark's VAE learns a noise level and adds it when sampling.

**Splits.**
- *ZTF, the paper's split (over curves):* 42% (CEP) to 85% (RRL) of the held-out curves belong to a star that also has a curve in training (the same star in the other band, or under a second object id). In our split over stars it is 0%.
- *Catalina:* every curve is its own object. 19% of the held-out EBSD/D curves share their period value with a training curve, but with 30,000 contact binaries between 0.2 and 0.5 days that is what chance gives, not the same star twice. Grouping by period is only the cautious choice there.
- *Resolution:* the paper tests on 36 curves per class on Catalina (one curve = 2.8 points; a recall of 75% has a standard error of 7.2 points) and 94 on ZTF (4.5 points). Our test set brings the common classes to 0.6–1.4 points. For the rarest classes it cannot do better than the data: 36 CEP and 49 DSCT test curves on Catalina, 96 and 89 on ZTF, so their recalls stay uncertain by 4–7 points on one split. Three splits and the paired bootstrap are how the benchmark deals with that.

**Which classes are confused** (one CNN, six epochs, not the benchmark's ensemble).
- Catalina: accuracy 88.8% but balanced accuracy 70.0%. Recall is 86–91% for the four larger classes, 59% for DSCT and **8% for CEP**: 64% of the Cepheids are called RR Lyrae and 17% contact binaries; 35% of the DSCT are called RR Lyrae.
- ZTF: accuracy 90.5%, balanced accuracy 65.2%. Recall 83–97% for EB, RRL, LPV, 45% for DSCT and **11.5% for CEP** (60% called RR Lyrae, 16% LPV).
- So the rare classes are not spread over all classes: they are absorbed by RR Lyrae, whose folded shape they resemble. This is where synthetic curves have room to help, and the per-class recall of CEP and DSCT is the number to watch in notebooks 03–06.
- On ZTF this single CNN is about 7 points below the paper's row A (72.6%), on Catalina about 3 below (73.4%). The paper's rows are ensembles on its own split; whether the larger gap on ZTF comes from the leakage above is a question for the `real_s` jobs of the benchmark, not answered here.

**The PELS-VAE file (OGLE-III).** 65,760 curves of 33,108 stars. The small classes exist mostly as copies: 70 curves per star for ACEP (71 stars), 121 for DSCT (42 stars), 12 for T2CEP. In their random split every validation star of ACEP, DSCT and T2CEP, and 98% of ELL, has a copy in training; overall 47.5%. Only 1.4% of the ACEP curves and 29% of the RR Lyrae have all three physical parameters, so the cVAE-P of notebook 02 is trained on a different mix of classes than the cVAE.